# Продвинутый прогноз выступлений Q2–Q4 2026

Ансамбль трёх моделей:
1. **Holt-Winters** — тренд + сезонность (выбор варианта по AIC)
2. **SARIMA** — авторегрессия + скользящее среднее + сезонность
3. **BSTS** — байесовский структурный тренд (фильтр Калмана)

Веса ансамбля определяются через **leave-last-1-out валидацию** (Q1 2026 — holdout).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from scipy.optimize import minimize
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from itertools import product

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 130
print('Библиотеки загружены.')

In [ ]:
quarters = [
    'Q1 2024', 'Q2 2024', 'Q3 2024', 'Q4 2024',
    'Q1 2025', 'Q2 2025', 'Q3 2025', 'Q4 2025',
    'Q1 2026',
]
raw = {
    'Выступления':         [1595, 8913, 16406, 27472, 9780, 18358, 10499, 28077, 12350],
    'Платные выступления': [1005, 5615, 10336, 17307, 5281,  9913,  5669, 15162,  5070],
    'Оплаты':              [ 365, 2384,  2970,  9668, 1077,  5548,  5212, 10210,  3761],
}

n_hist            = len(quarters)    # 9
n_val             = 1                # holdout = Q1 2026
n_trn             = n_hist - n_val   # 8 точек для валидации
forecast_quarters = ['Q2 2026', 'Q3 2026', 'Q4 2026']
all_quarters      = quarters + forecast_quarters

print('Исторические данные:')
print(pd.DataFrame(raw, index=quarters).to_string())

In [ ]:
def calc_rmse(actual, predicted):
    a = np.asarray(actual, float)
    p = np.asarray(predicted, float)
    mask = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[mask] - p[mask])**2))

def calc_mape(actual, predicted):
    a = np.asarray(actual, float)
    p = np.asarray(predicted, float)
    mask = ~(np.isnan(a) | np.isnan(p)) & (a != 0)
    return np.mean(np.abs((a[mask] - p[mask]) / a[mask])) * 100

def resid_ci(y, fitted, steps):
    a, f = np.asarray(y, float), np.asarray(fitted, float)
    mask = ~(np.isnan(a) | np.isnan(f))
    sigma = np.std(a[mask] - f[mask], ddof=1)
    return 1.96 * sigma * np.sqrt(np.arange(1, steps + 1))

In [ ]:
# МОДЕЛЬ 1 — Holt-Winters (перебор trend/seasonal, выбор по AIC)
def fit_holtwinters(y_arr, n_forecast=3):
    y    = np.asarray(y_arr, dtype=float)
    best = None
    if len(y) >= 2 * 4:
        for trend in ('add', 'mul'):
            for seasonal in ('add', 'mul'):
                try:
                    m = ExponentialSmoothing(
                        y, trend=trend, seasonal=seasonal,
                        seasonal_periods=4, damped_trend=True,
                        initialization_method='estimated'
                    ).fit(optimized=True)
                    if best is None or m.aic < best['aic']:
                        best = {'model': m, 'aic': m.aic,
                                'label': f'Holt-Winters ({trend}/{seasonal})'}
                except Exception:
                    pass
    if best is None:
        for trend in ('add', 'mul', None):
            try:
                m = ExponentialSmoothing(
                    y, trend=trend, seasonal=None,
                    damped_trend=(trend is not None),
                    initialization_method='estimated'
                ).fit(optimized=True)
                if best is None or m.aic < best['aic']:
                    best = {'model': m, 'aic': m.aic,
                            'label': f'Holt-Winters (trend={trend}/no-seasonal)'}
            except Exception:
                pass
    m  = best['model']
    fc = np.asarray(m.forecast(n_forecast), dtype=float)
    fv = np.asarray(m.fittedvalues, dtype=float)
    return {
        'fitted': fv, 'fc': fc, 'ci95': resid_ci(y, fv, n_forecast),
        'label': best['label'],
    }

print('Модель 1 (Holt-Winters) — определена.')

In [ ]:
# МОДЕЛЬ 2 — SARIMA (ограниченный поиск для коротких рядов)
def fit_sarima(y_arr, n_forecast=3):
    y    = np.asarray(y_arr, dtype=float)
    best = None
    for p, d, q in product(range(2), range(2), range(2)):
        for P, D, Q in [(0,0,0), (1,0,0), (0,0,1), (1,1,0), (0,1,1)]:
            try:
                m = SARIMAX(
                    y, order=(p, d, q),
                    seasonal_order=(P, D, Q, 4),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                ).fit(disp=False, maxiter=200)
                fc_test = np.asarray(m.get_forecast(n_forecast).predicted_mean, float)
                if np.any(np.isnan(fc_test)):
                    continue
                if best is None or m.aic < best['aic']:
                    best = {'model': m, 'aic': m.aic,
                            'order': (p,d,q), 'seasonal': (P,D,Q,4)}
            except Exception:
                pass
    m      = best['model']
    fc_res = m.get_forecast(steps=n_forecast)
    fc     = np.asarray(fc_res.predicted_mean, dtype=float)
    fv     = np.asarray(m.fittedvalues, dtype=float)
    try:
        ci_arr = np.asarray(fc_res.conf_int(alpha=0.05), dtype=float)
        ci95   = (ci_arr[:, 1] - ci_arr[:, 0]) / 2
        if np.any(np.isnan(ci95)):
            raise ValueError
    except Exception:
        ci95 = resid_ci(y, fv, n_forecast)
    return {
        'fitted': fv, 'fc': fc, 'ci95': np.abs(ci95),
        'label': f'SARIMA{best["order"]}x{best["seasonal"]}',
    }

print('Модель 2 (SARIMA) — определена.')

In [ ]:
# МОДЕЛЬ 3 — BSTS (фильтр Калмана: уровень + наклон + квартальная сезонность)
# State = [mu, beta, g1, g2, g3]
#   mu_t   = mu_{t-1} + beta_{t-1} + noise_level
#   beta_t = beta_{t-1}             + noise_slope
#   g1_t   = -g1-g2-g3 + noise_seasonal (квартальная сумма = 0)
#   y_t    = mu_t + g1_t + obs_noise
# fitted = one-step-ahead предсказания (честная мера ошибки)
def fit_bsts(y_arr, n_forecast=3):
    y = np.asarray(y_arr, dtype=float)
    T = len(y)
    F = np.array([
        [1, 1,  0,  0,  0],
        [0, 1,  0,  0,  0],
        [0, 0, -1, -1, -1],
        [0, 0,  1,  0,  0],
        [0, 0,  0,  1,  0],
    ], dtype=float)
    H = np.array([[1, 0, 1, 0, 0]], dtype=float)

    def kalman(log_vars, return_full=False):
        sv = np.exp(np.clip(log_vars, -20, 20))
        R  = np.array([[sv[0]**2]])
        Q  = np.diag([sv[1]**2, sv[2]**2, sv[3]**2, 0.0, 0.0])
        slope0 = (y[1] - y[0]) if T > 1 else 0.0
        x  = np.array([y[0], slope0, 0, 0, 0], dtype=float)
        P  = np.eye(5) * 1e4
        ll = 0.0
        xs, Ps, one_step = [], [], []
        for t in range(T):
            xp = F @ x;  Pp = F @ P @ F.T + Q
            S  = H @ Pp @ H.T + R
            inn = y[t] - (H @ xp)[0]
            K  = (Pp @ H.T) / S[0, 0]
            one_step.append((H @ xp)[0])
            x  = xp + K.ravel() * inn
            P  = (np.eye(5) - K @ H) @ Pp
            ll -= 0.5 * (np.log(2*np.pi) + np.log(abs(S[0,0])) + inn**2/S[0,0])
            xs.append(x.copy());  Ps.append(P.copy())
        if return_full:
            return ll, xs, Ps, Q, R, np.array(one_step)
        return -ll

    scale = np.std(y)
    x0 = np.log([scale*0.05, scale*0.02, scale*0.01, scale*0.05])
    opt = minimize(kalman, x0, method='Nelder-Mead',
                   options={'maxiter': 10000, 'xatol': 1e-8, 'fatol': 1e-8})
    _, xs, Ps, Q, R, one_step = kalman(opt.x, return_full=True)

    x = xs[-1].copy();  P = Ps[-1].copy()
    fc_vals, fc_vars = [], []
    for _ in range(n_forecast):
        x = F @ x;  P = F @ P @ F.T + Q
        fc_vals.append((H @ x)[0])
        fc_vars.append((H @ P @ H.T + R)[0, 0])

    return {
        'fitted': one_step,
        'fc':     np.array(fc_vals),
        'ci95':   1.96 * np.sqrt(np.array(fc_vars)),
        'label':  'BSTS (Kalman)',
    }

print('Модель 3 (BSTS/Kalman) — определена.')

In [ ]:
# АНСАМБЛЬ: leave-last-1-out валидация для весов
# Обучаем на первых 8 точках (Q1 2024 – Q4 2025),
# прогнозируем Q1 2026 → ошибка → вес = 1/|ошибка|
all_results = {}

for metric, y_arr in raw.items():
    y     = np.array(y_arr, dtype=float)
    y_trn = y[:n_trn]
    y_val = y[n_trn:]

    print(f'\n--- {metric} ---')

    val_rmse = {}
    for name, fn in [('HW', fit_holtwinters), ('SARIMA', fit_sarima), ('BSTS', fit_bsts)]:
        res_v  = fn(y_trn, n_forecast=n_val)
        fc_v   = np.asarray(res_v['fc'], float)
        err    = float(y_val[0] - fc_v[0])
        rmse_v = abs(err)
        val_rmse[name] = max(rmse_v, y.mean() * 0.01)
        print(f'  [{name}] прогноз={int(round(fc_v[0]))}  факт={int(y_val[0])}  ошибка={int(round(err))}')

    hw   = fit_holtwinters(y, n_forecast=3)
    sar  = fit_sarima(y, n_forecast=3)
    bsts = fit_bsts(y, n_forecast=3)
    models = [hw, sar, bsts]

    for m, name in zip(models, ['HW', 'SARIMA', 'BSTS']):
        m['val_rmse']  = val_rmse[name]
        m['rmse_hist'] = calc_rmse(y, m['fitted'])
        m['mape_hist'] = calc_mape(y, m['fitted'])

    inv_rmse = np.array([1.0 / m['val_rmse'] for m in models])
    weights  = inv_rmse / inv_rmse.sum()

    ens_fc = sum(w * m['fc']   for w, m in zip(weights, models))
    ens_ci = sum(w * m['ci95'] for w, m in zip(weights, models))
    spread = np.std([m['fc'] for m in models], axis=0)
    ens_ci = np.sqrt(ens_ci**2 + spread**2)

    all_results[metric] = {
        'models': models, 'weights': weights,
        'ens_fc': ens_fc, 'ens_ci': ens_ci, 'y': y,
    }

    wlbl = ', '.join(f'{n}={w:.2f}' for n, w in zip(['HW','SARIMA','BSTS'], weights))
    print(f'  Веса: {wlbl}')
    print(f'  Ансамбль Q2/Q3/Q4: {[int(round(v)) for v in ens_fc]}')

In [ ]:
print('=' * 80)
print('ПРОГНОЗ Q2-Q4 2026  (ансамбль: Holt-Winters + SARIMA + BSTS)')
print('=' * 80)
print(f'{"Метрика":<25} {"Квартал":<10} {"Прогноз":>9} {"От (-95%)":>11} {"До (+95%)":>11}')
print('-' * 80)
for metric, res in all_results.items():
    for i, q in enumerate(forecast_quarters):
        v  = float(res['ens_fc'][i])
        ci = float(res['ens_ci'][i])
        lo = max(0, int(round(v - ci)))
        hi = int(round(v + ci))
        print(f"{metric:<25} {q:<10} {int(round(v)):>9,} {lo:>11,} {hi:>11,}".replace(',', ' '))

In [ ]:
model_colors  = ['#F59E0B', '#8B5CF6', '#06B6D4']
metric_colors = {
    'Выступления':         '#2563EB',
    'Платные выступления': '#16A34A',
    'Оплаты':              '#DC2626',
}

fig, axes = plt.subplots(3, 1, figsize=(14, 16), sharex=True)
fig.suptitle('Ансамблевый прогноз Q2-Q4 2026\n(Holt-Winters + SARIMA + BSTS, веса по Q1 2026)',
             fontsize=14, fontweight='bold', y=0.995)

x_all  = np.arange(len(all_quarters))
x_hist = x_all[:n_hist]
x_fc   = x_all[n_hist:]

for ax, (metric, res) in zip(axes, all_results.items()):
    main_c = metric_colors[metric]
    y_obs  = res['y']

    for m, mc in zip(res['models'], model_colors):
        ax.plot(x_fc, np.maximum(0, m['fc']),
                '--', color=mc, lw=1.5, alpha=0.85, label=m['label'])

    ens = res['ens_fc']
    ci  = res['ens_ci']
    ax.plot(x_fc, ens, 'o-', color=main_c, lw=2.8, ms=9, zorder=5, label='Ансамбль')
    ax.fill_between(x_fc,
                    np.maximum(0, ens - ci), ens + ci,
                    color=main_c, alpha=0.15, label='95% ДИ')

    ax.scatter(x_hist, y_obs, color=main_c, s=70, zorder=6, label='Факт')
    ax.plot(x_hist, y_obs, '-', color=main_c, lw=1.2, alpha=0.5)

    for xi, yi in zip(x_fc, ens):
        ax.annotate(f'{int(round(yi)):,}'.replace(',', '\u202f'),
                    xy=(xi, yi), xytext=(0, 11), textcoords='offset points',
                    ha='center', fontsize=10, fontweight='bold', color=main_c)

    ax.axvline(x=n_hist - 0.5, color='gray', ls=':', lw=1.5)
    ylo, yhi = ax.get_ylim()
    ax.text(n_hist - 0.38, yhi * 0.96, 'прогноз ->', fontsize=8, color='gray', va='top')

    wlbl = ' | '.join(f'{n}={w:.2f}' for n, w in zip(['HW','SARIMA','BSTS'], res['weights']))
    ax.set_title(f'{metric}  [веса: {wlbl}]', fontsize=11, fontweight='bold', pad=6)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{int(x):,}'.replace(',', '\u202f'))
    )
    ax.grid(axis='y', alpha=0.3)
    ax.legend(loc='upper left', fontsize=8.5, framealpha=0.9, ncol=2)

axes[-1].set_xticks(x_all)
axes[-1].set_xticklabels(all_quarters, rotation=35, ha='right', fontsize=10)

plt.tight_layout()
plt.savefig('forecast_2026_advanced.png', bbox_inches='tight', dpi=150)
plt.show()
print('График сохранён: forecast_2026_advanced.png')

In [ ]:
summary = {}
for metric in raw:
    summary[metric] = list(raw[metric]) + [int(round(float(v))) for v in all_results[metric]['ens_fc']]

df_summary = pd.DataFrame(summary, index=all_quarters)
df_summary.index.name = 'Квартал'

print('\nИстория + прогноз (* -- прогнозные кварталы):')
hdr = f"{'':12}  {'Выступления':>14}  {'Платные':>14}  {'Оплаты':>10}"
print(hdr)
print('-' * len(hdr))
for i, (idx, row) in enumerate(df_summary.iterrows()):
    tag = '* ' if i >= n_hist else '  '
    print((f"{tag}{idx:<12}  "
           f"{row['Выступления']:>14,}  "
           f"{row['Платные выступления']:>14,}  "
           f"{row['Оплаты']:>10,}").replace(',', ' '))